**PHASE 1.** We first proceed with generating the rows of the tables before doing heavy math and generating computations.

**Step 1.** Setting up imports and global constants

In [1]:
# initialize imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# initialize constants
CONFIG = {
    "seed": 42,
    "n_locations": 10000,
    "n_events": 200
}

np.random.seed(CONFIG["seed"])

Locations rows generation

In [2]:
# create location id's
def generate_locations_base(n):
  df = pd.DataFrame({
      "location_id": np.arange(n)
  })
  return df

**Step 2.** Generating Locations

In [3]:
# assigning regions
def assign_regions(df):
  regions = ["Luzon", "Visayas", "Mindanao"]
  probs = [0.5, 0.2, 0.3]

  df["region"] = np.random.choice(regions, size=len(df), p=probs)
  return df

In [4]:
# generating longitude and latitude bounds for regions

# latitude and longitude ranges of the regions
REGION_BOUNDS = {
    "Luzon": {"lat": (12, 19), "lon": (116, 122)},
    "Visayas": {"lat": (9, 12), "lon": (120, 126)},
    "Mindanao": {"lat": (5, 9), "lon": (120, 127)}
}

# function that generates coordinates
def generate_coordinates(df):
  lat = []
  lon = []

  # iterates through the region column and assigns coordinates within those bounds
  for r in df["region"]:
    bounds = REGION_BOUNDS[r]
    lat.append(np.random.uniform(*bounds["lat"]))
    lon.append(np.random.uniform(*bounds["lon"]))

  # adds the latitude and longitude lists to the dataframe
  # coordinates are rounded to 4 decimal places to reflect barangay distances (1-5km across)
  df["lat"] = lat
  df["lon"] = lon

  # coordinates are rounded to 4 decimal places to reflect barangay distances (1-5km across)
  df["lat"] = df["lat"].round(4)
  df["lon"] = df["lon"].round(4)

  return df

In [5]:
locations = generate_locations_base(CONFIG["n_locations"])
locations = assign_regions(locations)
locations = generate_coordinates(locations)

 Elevation proxies row generation

In [6]:
# generate elevation proxy id's
def generate_elevation_table(locations):
  df = pd.DataFrame({
        "elevation_proxy_id": range(0, len(locations)),
        "location_id": locations["location_id"]
    })
  return df

In [7]:
elevation_proxies = generate_elevation_table(locations)
locations["elevation_proxy_id"] = elevation_proxies["elevation_proxy_id"]

 Rainfall events row generation

In [8]:
# generates rainfall event id's
def generate_rainfall_events(n):
  df = pd.DataFrame({
      "event_id": np.arange(n)
  })
  return df

In [9]:
# add dates to generated event
def assign_event_dates(df):

  # here, date starts at january 1 this year
  start_dates = pd.to_datetime("2026-01-01") + pd.to_timedelta(
      np.random.randint(0, 365, size=len(df)), unit = "D"
  )

  # create duration (1-4 days)
  duration = np.random.randint(1, 5, size=len(df))

  df["start_date"] = start_dates
  df["duration"] = duration
  df["end_date"] = df["start_date"] + pd.to_timedelta(duration, unit="D")

  return df

In [10]:
# assigning season
def assign_season(df):

  # for simplicity, the month the season is based on is the starting month
  month = df["start_date"].dt.month

  # wet season from june to november while dry season is december to may
  df["season"] = np.where(month.isin([6,7,8,9,10,11]), "wet", "dry")

  return df

In [11]:
events = generate_rainfall_events(CONFIG["n_events"])
events = assign_event_dates(events)
events = assign_season(events)

 Asset rows generation

In [12]:
# generate asset count for each location
def generate_asset_counts(locations):

  # temporary simple version
  avg_assets = 20

  counts = np.random.poisson(avg_assets, size=len(locations))
  locations["n_assets"] = counts

  return locations

In [13]:
# vectorized expansion into asset rows
def expand_assets_vectorized(locations):
  # repeat location_id based on n_assets
  location_ids = np.repeat(
      locations["location_id"].values,
      locations["n_assets"].values
  )

  df = pd.DataFrame({
      "asset_id": np.arange(len(location_ids)),
      "location_id": location_ids
  })

  return df

In [14]:
locations = generate_asset_counts(locations)
assets = expand_assets_vectorized(locations)

 Damage table rows generation

In [15]:
# generate report id's
def generate_damage_base(assets):
  df = pd.DataFrame({
      "damage_id": np.arange(len(assets)),
      "asset_id": assets["asset_id"]
  })
  return df

In [16]:
damages = generate_damage_base(assets)

**PHASE 2.** We will now translate the mathematical model into code.

**Step 3.** Generate socioeconomic variables in the locations table

**Step 3.1.** Urbanization level

In [17]:
# define urban centers (manila, cebu, davao)
URBAN_CENTERS = {
    "Manila": {"lat": 14.6, "lon": 121.0, "weight": 1.0},
    "Cebu": {"lat": 10.3, "lon": 123.9, "weight": 0.6},
    "Davao": {"lat": 7.1, "lon": 125.6, "weight": 0.5}
}

# distance function
def compute_distance(lat1, lon1, lat2, lon2):
  return np.sqrt((lat1 - lat2)**2 + (lon1 - lon2)**2)

# computing urbanization level score
def generate_urban(locations, sigma=2.0):

  # put zeroes as a placeholder for every location
  urban_score = np.zeros(len(locations))

  # compute the distance to the urban centers for all locations
  for region, center in URBAN_CENTERS.items():
    d = compute_distance(
        locations["lat"].values,
        locations["lon"].values,
        center["lat"],
        center["lon"]
    )

    # computing the function that is inside the summation of the formula
    influence = center["weight"] * np.exp(-(d**2)/(2 * sigma**2))

    # sum the influences
    urban_score += influence

  # adding noise
  noise = np.random.normal(0, 0.1, size=len(locations))
  urban_level = urban_score + noise

  # clipping to [0,1]
  locations["urban_level"] = np.clip(urban_level, 0, 1)

  return locations

In [18]:
locations = generate_urban(locations)
locations["urban_level"].describe()

,urban_level
count,10000.000000
mean,0.406456
std,0.274574
min,0.000000
25%,0.168338
50%,0.395924
75%,0.617980
max,1.000000


**Step 3.2.** Population

In [19]:
def generate_population(locations):

  # apply the lognormal distribution to the population
  pop = np.random.lognormal(mean=8.01, sigma=1.2, size=len(locations))

  # add noise
  noise = np.random.normal(0, 0.1, size=len(locations))
  pop = pop * np.exp(noise)

  # cap at max baranggay size
  pop = np.clip(pop, 0, 261729)

  # round to integers
  locations["population"] = np.round(pop).astype(int)

  return locations

locations = generate_population(locations)

In [20]:
locations["population"].describe()

,population
count,10000.000000
mean,6051.773900
std,10607.269151
min,32.000000
25%,1291.000000
50%,2948.500000
75%,6616.500000
max,261729.000000


**Step 3.3.** Wealth Index

In [21]:
def generate_wealth(locations):

  # normalize population first
  pop_scaled = locations["population"]/3000

  # add noise
  noise = np.random.normal(0, 0.1, size=len(locations))

  # compute wealth based on the model
  wealth = (
      0.5 * locations["urban_level"] +
      0.2 * pop_scaled +
      0.3 * noise
  )

  # clip values and create the wealth column
  locations["wealth_index"] = np.clip(wealth, 0, 1)

  return locations

locations = generate_wealth(locations)

**Step 3.4.** Vegetation Index

In [22]:
def generate_vegetation(locations):

  # add noise
  noise = np.random.normal(0, 0.3, size=len(locations))

  # compute vegetation based on the model
  vegetation = 1 - locations["urban_level"] + noise

  # clip the value and create the vegetation column
  locations["vegetation_index"] = np.clip(vegetation, 0, 1)

  return locations

locations = generate_vegetation(locations)

In [23]:
# checking correlation
locations[["urban_level", "vegetation_index"]].corr()

# strong negative correlation of -0.68 which is alright

,urban_level,vegetation_index
urban_level,1.000000,-0.677266
vegetation_index,-0.677266,1.000000


**Step 4.** Add variables to the elevation proxies table.

**Step 4.1.** Elevation

In [24]:
def generate_elevation(elevation_df, locations):

  # multiplier for frequency of mountainous terrains
  freq_map = {
      "Luzon": 0.25,
      "Visayas": 0.10,
      "Mindanao": 0.15
  }

  # amplifier for elevation based on region
  amp_map = {
      "Luzon": 6,
      "Visayas": 5,
      "Mindanao": 6
  }

  # align location data
  loc = locations.set_index("location_id").loc[elevation_df["location_id"]]

  # assign variable for freq_map and amp_map
  f = loc["region"].map(freq_map).values
  A = loc["region"].map(amp_map).values

  # get the coordinates
  lat = loc["lat"].values
  lon = loc["lon"].values

  # main elevation formula
  elevation = A * 500 * np.abs(
      np.sin(f * lat) * np.cos(f * lon)
  )

  # assign noise
  noise = np.random.normal(0, 10, size=len(elevation_df))

  # create the elevation column
  elevation_df["elevation"] = np.clip(elevation + noise, 0, None)

  return elevation_df

**Step 4.2.** Distance to water

In [25]:
def generate_distance_to_water(elevation_df, scale=1000, d_max=10000):

  # sample exponential
  dist = np.random.exponential(scale=scale, size=len(elevation_df))

  # truncate (via clipping)
  dist = np.clip(dist, 0, d_max)

  # create the column
  elevation_df["distance_to_water"] = dist

  return elevation_df

**Step 4.3.** Slope

In [26]:
def generate_slope(elevation_df):

  # gamma(2,5) of the model
  gamma_part = np.random.gamma(shape=2, scale=5, size=len(elevation_df))

  # scale the elevation first
  elevation_scaled = elevation_df["elevation"] / 3000

  # noise
  noise = np.random.normal(0, 2, size=len(elevation_df))

  # main slope formula
  slope = gamma_part + 20 * elevation_scaled + noise

  # create the slope column
  elevation_df["slope"] = np.clip(slope, 0, 45)

  return elevation_df

**Step 4.4** Flood Susceptibility

In [27]:
# sigmoid function
def sigmoid(x):
  return 1 / (1 + np.exp(-x))

# main function for generating flood susceptibility
def generate_flood_sus(elevation_df):

  # scale the computed variables earlier
  slope_scaled = elevation_df["slope"] / 45
  elevation_scaled = elevation_df["elevation"] / 3000
  distance_scaled = elevation_df["distance_to_water"] / 1000

  # random noise
  noise = np.random.normal(0, 0.1 + 0.05 * (1 - elevation_scaled), size=len(elevation_df))

  # the linear equation that would be put in the sigmoid function later
  linear = (
      -0.3 * slope_scaled
      -0.4 * elevation_scaled
      -0.3 * distance_scaled
      + 0.8
      + noise
  )

  # get the actual susceptibility
  flood_sus = sigmoid(linear)

  # create the column
  elevation_df["flood_susceptibility"] = np.clip(flood_sus, 0, 1)

  return elevation_df

In [28]:
elevation_proxies = generate_elevation(elevation_proxies, locations)
elevation_proxies = generate_distance_to_water(elevation_proxies)
elevation_proxies = generate_slope(elevation_proxies)
elevation_proxies = generate_flood_sus(elevation_proxies)

**Step 5.** Latent Variables (flood control and soil drainage)

**Step 5.1.** Flood Control Capacity

In [29]:
def generate_flood_control(locations):

  # noise
  noise = np.random.normal(0, 0.1, size=len(locations))

  # main flood control equation
  fc = (
      0.5 * locations["wealth_index"] +
      0.3 * locations["urban_level"] +
      noise
  )

  # create the column
  locations["flood_control"] = np.clip(fc, 0, 1)

  return locations

**Step 5.2.** Soil Drainage

In [30]:
def generate_soil_drainage(locations):

  # noise
  noise = np.random.normal(0, 0.1, size=len(locations))

  # main equation
  sd = (
      0.7 * locations["vegetation_index"] +
      0.2 * (1 - locations["urban_level"]) +
      noise
  )

  # create column
  locations["soil_drainage"] = np.clip(sd, 0, 1)

  return locations

In [31]:
locations = generate_vegetation(locations)

locations = generate_flood_control(locations)
locations = generate_soil_drainage(locations)

**Event and Location Interaction**

here, we pair rainfall events and the locations in which they occur.

Generate event location pairs

In [32]:
def generate_event_location_pairs(locations, events):

  # assign regional probabilities of rainfall events
  region_prob = {
      "Luzon": 0.25,
      "Visayas": 0.20,
      "Mindanao": 0.15,
  }

  # placeholder for rows
  all_rows = []

  # iterate through the events
  for _, event in events.iterrows():

    # event intensity
    event_intensity = np.random.beta(2, 2)

    # create an independent copy of locations
    loc_copy  = locations.copy()

    # assign exposure probability by region
    loc_copy["exposure_p"] = loc_copy["region"].map(region_prob) * event_intensity

    loc_copy["exposure_p"] = loc_copy["exposure_p"].clip(0, 1)

    # sample affected locations
    mask = np.random.uniform(0, 1, len(loc_copy)) < loc_copy["exposure_p"]

    affected = loc_copy[mask].copy()
    affected["event_id"] = event["event_id"]

    all_rows.append(affected[["location_id", "event_id"]])

  df = pd.concat(all_rows, ignore_index=True)

  return df

In [33]:
event_location = generate_event_location_pairs(locations, events)

**Step 6.** Generate Rainfall Events

In [34]:
# add an event region
def assign_event_region(events):

  # the regions
  regions = ["Luzon", "Visayas", "Mindanao"]
  # same probabilities
  probs = [0.5, 0.2, 0.3]

  events["event_region"] = np.random.choice(regions, size=len(events), p=probs)

  return events

events = assign_event_region(events)

# generate rainfal event
def generate_rainfall(events):

  # assign base rainfall strength based on region
  base_rain = {
      "Luzon": {"dry": 100, "wet": 300},
      "Visayas": {"dry": 80, "wet": 250},
      "Mindanao": {"dry": 120, "wet": 200}
  }

  # placeholder for rainfall
  rainfall = []

  # iterate through the rows
  for _, row in events.iterrows():

    # get the region and season
    region = row["event_region"]
    season = row["season"]

    # get the base rain from the region and season
    base = base_rain[region][season]

    # noise
    noise = np.random.normal(0, 0.2)

    # actual rainfall with added noise
    value = base * np.exp(noise)

    rainfall.append(value)

  # make the rainfall list the rainfall column
  events["rainfall_mm"] = rainfall

  return events

events = generate_rainfall(events)

Merge rainfall events with event-location table

In [35]:
# merge the rainfall intensities with the corresponding events
event_location = event_location.merge(
    events[["event_id", "rainfall_mm"]],
    on="event_id",
    how="left"
)

# merge the flood susceptibilitis with the corresponding locations
event_location = event_location.merge(
    elevation_proxies[[
        "location_id",
        "flood_susceptibility"
    ]],
    on="location_id",
    how="left"
)

# merge the flood control and soil drainage with the corresponding locations
event_location = event_location.merge(
    locations[[
        "location_id",
        "flood_control",
        "soil_drainage"
    ]],
    on="location_id",
    how="left"
)

event_location.shape

(220517, 6)

**Steo 7.** Model flood risk

In [36]:
def generate_flood_risk(df):

  # noise
  noise = np.random.normal(0, 0.1, size=len(df))

  # main risk formula
  risk = (
      0.4 * (df["rainfall_mm"] / 300) +
      0.3 * df["flood_susceptibility"] -
      0.2 * df["flood_control"] -
      0.1 * df["soil_drainage"] +
      noise
  )

  # create the column
  df["flood_risk"] = np.clip(risk, 0, 1)

  return df

event_location = generate_flood_risk(event_location)

**Step 8.** Generate asset variables

**Step 8.1.** Generate asset type

In [37]:
def assign_asset_types(assets):

  # probabilities of different asset types
  asset_types = ["residential", "commercial", "infrastructure", "vehicle"]
  type_probs = [0.5, 0.2, 0.2, 0.1]

  # sample from asset types
  assets["asset_type"] = np.random.choice(
      asset_types,
      size=len(assets),
      p=type_probs
  )

  return assets

**Step 8.2.** Generate asset substype

In [38]:
def assign_asset_subtypes(assets):

  # table of asset type and subtypes with probabilities
  subtype_map = {
      "residential": (["low", "mid", "high"], [0.5, 0.35, 0.15]),
      "commercial": (["small", "medium", "large"], [0.7, 0.25, 0.05]),
      "infrastructure": (["local", "major", "critical"], [0.75, 0.2, 0.05]),
      "vehicle": (["motorcycle", "car", "PUV"], [0.6, 0.3, 0.1])
  }

  # subtype placeholder
  subtypes = []

  # iterate through every asset and check the type
  for t in assets["asset_type"]:
    choices, probs = subtype_map[t]
    subtypes.append(np.random.choice(choices, p=probs))

  # create the asset subtype column
  assets["asset_subtype"] = subtypes

  return assets

**Step 8.3.** Assign base asset values

In [39]:
BASE_VALUES = {
    ("residential", "low"): 1000000,
    ("residential", "mid"): 3000000,
    ("residential", "high"): 8000000,
    ("commercial", "small"): 5000000,
    ("commercial", "medium"): 50000000,
    ("commercial", "large"): 500000000,
    ("infrastructure", "local"): 10000000,
    ("infrastructure", "major"): 300000000,
    ("infrastructure", "critical"): 1000000000,
    ("vehicle", "motorcycle"): 100000,
    ("vehicle", "car"): 800000,
    ("vehicle", "PUV"): 500000
}

**Step 8.4.** Compute asset values

In [40]:
def generate_asset_values(assets, locations, beta_u=0.5):

  # merge urban level
  assets = assets.merge(
      locations[["location_id", "urban_level"]],
      on="location_id",
      how='left'
  )

  # base value lookup
  base_vals = []
  for t, s in zip(assets["asset_type"], assets["asset_subtype"]):
    base_vals.append(BASE_VALUES[(t,s)])

  base_vals = np.array(base_vals)

  # noise
  noise = np.random.normal(0, 0.12, size=len(assets))

  # main formula for asset values
  value = (
      base_vals *
      np.exp(beta_u * assets["urban_level"]) *
      np.exp(noise)
  )

  assets["asset_value"] = value

  return assets

In [41]:
assets = assign_asset_types(assets)
assets = assign_asset_subtypes(assets)
assets = generate_asset_values(assets, locations)

**Step 9** Damage generation

**Step 9.1.** Build asset-event table

In [42]:
# merge the assets with the event-location table
asset_event = event_location.merge(
    assets[["asset_id", "location_id", "asset_value"]],
    on="location_id",
    how="left"
)

# add damage_id
asset_event["damage_id"] = np.arange(len(asset_event))

asset_event.shape

(4411263, 10)

In [43]:
# mild shock heterogeneity
asset_event["event_intensity"] = np.random.gamma(shape=2, scale=0.5, size=len(asset_event))
asset_event["flood_risk"] *= asset_event["event_intensity"]

**Step 9.2.** Damage occurence

In [44]:
def generate_damage_occurence(df, threshold=0.4, gamma=15):

  # sigmoid function
  def sigmoid(x):
    return 1 / (1 + np.exp(-x))

  # generate probability
  prob = sigmoid(gamma * (df["flood_risk"] - threshold))
  prob[df["flood_risk"] < threshold] = 0

  # apply bernoulli
  rand = np.random.uniform(0, 1, len(df))
  df["damage_occurs"] = (rand < prob).astype(int)

  return df

asset_event = generate_damage_occurence(asset_event)

**Step 9.3.** Damage values

In [45]:
def generate_damage_value(df):

  # noise
  noise = np.random.normal(0, 0.1, size=len(df))

  # main damage equation
  damage = (
      df["asset_value"] *
      df["flood_risk"] *
      (1 - df["flood_control"]) *
      np.exp(noise)
  )

  # generate damage values only if there is damage
  df["damage_value"] = damage * df["damage_occurs"]

  return df

asset_event = generate_damage_value(asset_event)

**Step 9.4.** Generate damage level

In [46]:
def generate_damage_level(df):

  # get only those where there are damages
  damaged = df[df["damage_value"] > 0]["damage_value"]

  # assign quartiles
  q1 = damaged.quantile(0.33)
  q2 = damaged.quantile(0.66)

  # state conditions
  conditions = [
      df["damage_value"] == 0,
      df["damage_value"] <= q1,
      df["damage_value"] <= q2
  ]

  # possible damage levels
  choices = ["None", "Minor", "Moderate"]

  df["damage_level"] = np.select(
      conditions,
      choices,
      default="Severe"
  )

  return df

asset_event = generate_damage_level(asset_event)

**Step 9.5.** Create the damages table.

In [47]:
damages = asset_event[[
    "damage_id",
    "asset_id",
    "event_id",
    "damage_occurs",
    "damage_value",
    "damage_level"
]].copy()

**Step 10.** Creating the tables to be exported and dropping latent variables.

In [48]:
cols_to_drop = [
    "soil_drainage",
    "flood_control",
    "flood_risk",
    "event_intensity"  # also latent-like shock term
]

asset_event = asset_event.drop(columns=cols_to_drop, errors="ignore")
event_location = event_location.drop(columns=cols_to_drop, errors="ignore")

# locations table
locations = locations[[
    "location_id",
    "region",
    "lat",
    "lon",
    "urban_level",
    "population",
    "wealth_index",
    "vegetation_index",
    "elevation_proxy_id"
]].copy()

# elevation proxies
elevation_proxies = elevation_proxies[[
    "elevation_proxy_id",
    "location_id",
    "elevation",
    "distance_to_water",
    "slope",
    "flood_susceptibility"
]].copy()

# events
events_final = event_location[[
    "event_id",
    "location_id",
    "rainfall_mm",
    "flood_susceptibility"
]].copy()

events = events_final.merge(
    events[[
        "event_id",
        "start_date",
        "end_date",
        "duration",
        "season",
        "event_region"
    ]],
    on="event_id",
    how="left"
)

# assets
assets  = assets[[
    "asset_id",
    "location_id",
    "asset_type",
    "asset_subtype",
    "asset_value"
]].copy()

**Step 11** Exporting the datasets

In [49]:
import os

OUTPUT_DIR = "synthetic_data_outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)

locations.to_csv(f"{OUTPUT_DIR}/locations.csv", index=False)
elevation_proxies.to_csv(f"{OUTPUT_DIR}/elevation_proxies.csv", index=False)
events.to_csv(f"{OUTPUT_DIR}/events.csv", index=False)
assets.to_csv(f"{OUTPUT_DIR}/assets.csv", index=False)
damages.to_csv(f"{OUTPUT_DIR}/damages.csv", index=False)

for name in ["locations", "elevation_proxies", "events", "assets", "damages"]:
    df = pd.read_csv(f"{OUTPUT_DIR}/{name}.csv")
    print(name, df.shape)



locations (10000, 9)
elevation_proxies (10000, 6)
events (220517, 9)
assets (200074, 5)
damages (4411263, 6)
